This script samples elevation data across a defined bounding box covering Columbia, South Carolina. It uses the USGS Elevation Point Query Service (EPQS) to retrieve elevation values in meters for a grid of latitude/longitude coordinates.

In [7]:
# requirements: pip install requests rasterio numpy affine tqdm geopandas
import os
import tempfile
import requests
import rasterio
from rasterio.warp import reproject, Resampling, calculate_default_transform
import numpy as np
from tqdm import tqdm

# === CONFIG ===
# Put your key into an env var before running:
# export OPENTOPO_API_KEY="your_real_key_here"
API_KEY = os.getenv("OPENTOPO_API_KEY")
if not API_KEY:
    raise SystemExit("Set OPENTOPO_API_KEY in your environment (do NOT hard-code keys).")

# Choose datasets:
DSM_DEMTYPE = "COP30"     # Copernicus Global DSM 30m (surface)
DEM_DEMTYPE = "NASADEM"   # NASADEM (terrain). You may choose SRTMGL1, etc.

# bounding box (example): small test box (replace with your area)
west, south, east, north = -81.1150, 33.9200, -80.9000, 34.1200

BASE_URL = "https://portal.opentopography.org/API/globaldem"

def download_dem(demtype, west, south, east, north, out_path, api_key):
    params = {
        "demtype": demtype,
        "west": west, "south": south, "east": east, "north": north,
        "outputFormat": "GTiff",
        "API_Key": api_key
    }
    print(f"Requesting {demtype} from OpenTopography...")
    r = requests.get(BASE_URL, params=params, stream=True, timeout=120)
    r.raise_for_status()
    # Save response to file
    total = int(r.headers.get('content-length', 0))
    with open(out_path, "wb") as f, tqdm(total=total, unit='B', unit_scale=True) as pbar:
        for chunk in r.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk); pbar.update(len(chunk))
    print("Saved to", out_path)

def align_to(src_path, target_meta):
    """Return an array aligned to target_meta (affine, width/height, crs)."""
    with rasterio.open(src_path) as src:
        dst_transform = target_meta["transform"]
        dst_crs = target_meta["crs"]
        dst_width = target_meta["width"]
        dst_height = target_meta["height"]
        dst = np.empty((dst_height, dst_width), dtype=np.float32)
        reproject(
            source=rasterio.band(src, 1),
            destination=dst,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=dst_transform,
            dst_crs=dst_crs,
            resampling=Resampling.bilinear,
            dst_nodata=np.nan
        )
    return dst

# === pipeline ===
with tempfile.TemporaryDirectory() as tmp:
    dsm_path = os.path.join(tmp, "dsm.tif")
    dem_path = os.path.join(tmp, "dem.tif")
    out_canopy = os.path.join(tmp, "canopy_height.tif")
    # Download DSM (surface)
    download_dem(DSM_DEMTYPE, west, south, east, north, dsm_path, API_KEY)
    # Download DEM (terrain)
    download_dem(DEM_DEMTYPE, west, south, east, north, dem_path, API_KEY)

    # Open DSM to define target grid
    with rasterio.open(dsm_path) as dsm_src:
        target_meta = dsm_src.meta.copy()
        dsm_data = dsm_src.read(1).astype(np.float32)
        dsm_nodata = dsm_src.nodata if dsm_src.nodata is not None else np.nan

    # Align/resample DEM to DSM grid
    dem_aligned = align_to(dem_path, target_meta)

    # Convert nodata values to nan for safe arithmetic
    dsm = np.where(np.isclose(dsm_data, dsm_nodata), np.nan, dsm_data)
    dem = np.where(np.isclose(dem_aligned, dsm_nodata), np.nan, dem_aligned)

    # canopy height (surface - ground)
    canopy = dsm - dem

    # Optional: threshold to remove spurious negatives / very small values
    canopy_mask = ~np.isnan(canopy) & (canopy > 0.5)  # keep heights > 0.5 m
    canopy_filtered = np.where(canopy_mask, canopy, np.nan)

    # Stats
    valid = canopy_filtered[~np.isnan(canopy_filtered)]
    if valid.size:
        print(f"Canopy stats (m): min {valid.min():.2f}, mean {valid.mean():.2f}, max {valid.max():.2f}")
    else:
        print("No canopy pixels found in AOI with threshold.")

    # Save canopy raster
    out_meta = target_meta.copy()
    out_meta.update(dtype=rasterio.float32, count=1, nodata=np.nan)
    with rasterio.open(out_canopy, "w", **out_meta) as dst:
        dst.write(canopy_filtered.astype(rasterio.float32), 1)
    print("Canopy height raster written to:", out_canopy)
    # If you want the file on disk long-term, move it from tmp or change out path accordingly.

Requesting COP30 from OpenTopography...


2.33MB [00:00, 2.47MB/s]


Saved to C:\Users\syder\AppData\Local\Temp\tmpyn1vu5td\dsm.tif
Requesting NASADEM from OpenTopography...


410kB [00:00, 790kB/s] 


Saved to C:\Users\syder\AppData\Local\Temp\tmpyn1vu5td\dem.tif
Canopy stats (m): min 0.50, mean 5.71, max 67.50
Canopy height raster written to: C:\Users\syder\AppData\Local\Temp\tmpyn1vu5td\canopy_height.tif


In [18]:
import os

output_dir = "data"
os.makedirs(output_dir, exist_ok=True)

dsm_path = os.path.join(output_dir, "columbia_dsm.tif")
dem_path = os.path.join(output_dir, "columbia_dem.tif")
canopy_path = os.path.join(output_dir, "columbia_canopy_height.tif")

In [19]:
download_dem(DSM_DEMTYPE, west, south, east, north, dsm_path, API_KEY)
download_dem(DEM_DEMTYPE, west, south, east, north, dem_path, API_KEY)

Requesting COP30 from OpenTopography...


2.33MB [00:00, 3.38MB/s]


Saved to data\columbia_dsm.tif
Requesting NASADEM from OpenTopography...


410kB [00:00, 1.35MB/s]

Saved to data\columbia_dem.tif


In [20]:
with rasterio.open(canopy_path, "w", **out_meta) as dst:
    dst.write(canopy_filtered.astype(rasterio.float32), 1)

print("Saved canopy height to:", canopy_path)

Saved canopy height to: data\columbia_canopy_height.tif
